In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install imbalanced-learn -q
print("安裝完成")


安裝完成


In [ ]:
import pandas as pd
import numpy as np
import glob, os, json, zipfile, tempfile, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, recall_score
from sklearn.cluster import DBSCAN
from imblearn.over_sampling import SMOTE

print("套件載入完成")

套件載入完成


In [ ]:
DATA_DIR = '/content/drive/MyDrive/TrafficRisk'

print("Step 1：讀取資料...")
all_dfs = []
tmpdir = tempfile.mkdtemp()

for z in glob.glob(os.path.join(DATA_DIR, '*.zip')):
    try:
        with zipfile.ZipFile(z, 'r') as zf:
            zf.extractall(tmpdir)
    except:
        pass

all_csv = (
    glob.glob(os.path.join(tmpdir, '**/*.csv'), recursive=True) +
    glob.glob(os.path.join(DATA_DIR, '*.csv'))
)

for path in all_csv:
    fname = os.path.basename(path).lower()
    if any(s in fname for s in ['manifest','schema','file.csv']):
        continue
    for enc in ['utf-8-sig', 'cp950']:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            if '發生地點' in df.columns and '經度' in df.columns:
                all_dfs.append(df)
                print(f"  ✓ {os.path.basename(path)}（{len(df):,}筆）")
                break
        except:
            continue

df_raw = pd.concat(all_dfs, ignore_index=True)
print(f"\n總共：{len(df_raw):,} 筆")

Step 1：讀取資料...
  ✓ 114年度A2交通事故資料_2.csv（66,239筆）
  ✓ 2020年度A2交通事故資料_7.csv（69,486筆）
  ✓ 114年度A2交通事故資料_1.csv（81,564筆）
  ✓ 111年度A2交通事故資料_9.csv（68,730筆）
  ✓ NPA_TMA2_1.csv（80,598筆）
  ✓ 2020年度A2交通事故資料_5.csv（65,887筆）
  ✓ 112年度A2交通事故資料_5.csv（76,352筆）
  ✓ 113年度A2交通事故資料_4.csv（68,747筆）
  ✓ 113年度A2交通事故資料_12.csv（84,558筆）
  ✓ 111年度A2交通事故資料_1.csv（77,289筆）
  ✓ 112年度A2交通事故資料_7.csv（75,705筆）
  ✓ 113年度A2交通事故資料_7.csv（73,163筆）
  ✓ 111年度A2交通事故資料_5.csv（60,963筆）
  ✓ 112年度A1交通事故資料.csv（4,495筆）
  ✓ 114年度A2交通事故資料_7.csv（71,584筆）
  ✓ 111年度A2交通事故資料_6.csv（61,823筆）
  ✓ NPA_TMA2_3.csv（73,889筆）
  ✓ 2021年度A2交通事故資料_6.csv（46,838筆）
  ✓ 113年度A2交通事故資料_8.csv（72,857筆）
  ✓ 114年度A2交通事故資料_8.csv（74,139筆）
  ✓ 113年度A2交通事故資料_6.csv（73,269筆）
  ✓ 2021年度A2交通事故資料_3.csv（68,318筆）
  ✓ 113年度A2交通事故資料_11.csv（76,692筆）
  ✓ 111年度A2交通事故資料_10.csv（72,892筆）
  ✓ 2021年度A2交通事故資料_9.csv（64,960筆）
  ✓ 2021年度A2交通事故資料_12.csv（82,524筆）
  ✓ 2020年度A2交通事故資料_1.csv（70,915筆）
  ✓ 2021年度A2交通事故資料_11.csv（74,157筆）
  ✓ 113年度A2交通事故資料_5.csv（72,537筆）
  ✓ 2020年度A2交通事故資料_8.csv（66,

In [ ]:
print("Step 2：篩選桃園市")
mask = (
    df_raw['處理單位名稱警局層'].str.contains('桃園', na=False) |
    df_raw['發生地點'].str.contains('桃園市', na=False) |
    df_raw['發生地點'].str.contains('桃園縣', na=False)
)
df = df_raw[mask].copy()
print(f"桃園市事故資料：{len(df):,} 筆")

# 限縮到中壢
TARGET_DISTRICTS = ['中壢區']
mask_dist = df['發生地點'].str.contains('|'.join(TARGET_DISTRICTS), na=False)
df = df[mask_dist].copy()
print(f"中壢 ：{len(df):,} 筆")

print("\nStep 3：清洗資料...")
df['經度'] = pd.to_numeric(df['經度'], errors='coerce')
df['緯度'] = pd.to_numeric(df['緯度'], errors='coerce')
df = df[
    df['緯度'].notna() & df['經度'].notna() &
    (df['緯度'] >= 24.92) & (df['緯度'] <= 25.02) &
    (df['經度'] >= 121.19) & (df['經度'] <= 121.30)
].copy()
print(f"清洗後：{len(df):,} 筆")

def get_severity(row):
    s = str(row.get('死亡受傷人數',''))
    if '死亡' in s:
        try:
            deaths = int(s.split('死亡')[1].split(';')[0])
            if deaths >= 1: return 2
        except: pass
    if 'A1' in str(row.get('事故類別名稱','')): return 2
    return 1

df['嚴重度'] = df.apply(get_severity, axis=1)
print(f"\nA1（死亡）：{(df['嚴重度']==2).sum():,} 筆")
print(f"A2（傷亡）：{(df['嚴重度']==1).sum():,} 筆")

def get_period(t):
    try:
        h = int(str(int(t)).zfill(6)[:2])
        if 7  <= h < 10: return '早峰'
        if 10 <= h < 17: return '日間'
        if 17 <= h < 20: return '晚峰'
        if 20 <= h < 24: return '夜間'
        return '深夜'
    except:
        return '日間'

df['時段'] = df['發生時間'].apply(get_period)
df['天候'] = df['天候名稱'].apply(
    lambda x: '雨天' if '雨' in str(x) else (
              '陰天' if '陰' in str(x) or '霧' in str(x) else '晴天')
)
df['有號誌'] = df['號誌-號誌種類名稱'].apply(
    lambda x: 0 if '無' in str(x) else 1
)
df['速限'] = pd.to_numeric(df['速限-第1當事者'], errors='coerce').fillna(50)
df['grid_lat'] = (df['緯度'] * 2000).round() / 2000
df['grid_lng'] = (df['經度'] * 2000).round() / 2000
df['發生年度_num'] = pd.to_numeric(df['發生年度'], errors='coerce')

def is_moto(row):
    col = '當事者區分-類別-大類別名稱-車種'
    if col in row and '機車' in str(row[col]):
        return True
    return False

df['涉及機車'] = df.apply(is_moto, axis=1)
print(f"涉及機車事故：{df['涉及機車'].sum():,} 筆（{df['涉及機車'].mean()*100:.1f}%）")

# 順便確認肇因欄位名稱
cause_cols = [c for c in df.columns if '肇因' in c]
print(f"\n肇因相關欄位：{cause_cols}")

print(f"\n時段分布：\n{df['時段'].value_counts()}")
print(f"\n天候分布：\n{df['天候'].value_counts()}")

Step 2：篩選桃園市
桃園市事故資料：659,564 筆
中壢 ：153,323 筆

Step 3：清洗資料...
清洗後：146,561 筆

A1（死亡）：329 筆
A2（傷亡）：146,232 筆
涉及機車事故：82,599 筆（56.4%）

肇因相關欄位：['肇因研判大類別名稱-主要', '肇因研判子類別名稱-主要', '肇因研判大類別名稱-個別', '肇因研判子類別名稱-個別']

時段分布：
時段
日間    55190
晚峰    31485
早峰    30412
夜間    19673
深夜     9801
Name: count, dtype: int64

天候分布：
天候
晴天    101460
陰天     23199
雨天     21902
Name: count, dtype: int64


In [ ]:
print("Step 4：以路口網格聚合（50×50公尺）...")

global_rain_rate  = (df['天候'] == '雨天').mean()
global_eve_rate   = (df['時段'] == '晚峰').mean()
global_night_rate = (df['時段'] == '夜間').mean()
global_peak_rate  = (df['時段'] == '早峰').mean()
global_moto_rate  = df['涉及機車'].mean()

print(f"全市平均雨天比例：{global_rain_rate:.3f}")
print(f"全市平均晚峰比例：{global_eve_rate:.3f}")
print(f"全市平均夜間比例：{global_night_rate:.3f}")
print(f"全市平均早峰比例：{global_peak_rate:.3f}")
print(f"全市平均機車比例：{global_moto_rate:.3f}")

grid = df.groupby(['grid_lat','grid_lng']).agg(
    事故總數=('嚴重度', 'count'),
    A1件數=('嚴重度', lambda x: (x==2).sum()),
    A2件數=('嚴重度', lambda x: (x==1).sum()),
    雨天比例=('天候', lambda x: (x=='雨天').mean()),
    有號誌=('有號誌', 'mean'),
    平均速限=('速限', 'mean'),
    晚峰比例=('時段', lambda x: (x=='晚峰').mean()),
    夜間比例=('時段', lambda x: (x=='夜間').mean()),
    早峰比例=('時段', lambda x: (x=='早峰').mean()),
    機車比例=('涉及機車', 'mean'),
).reset_index()

grid = grid[grid['事故總數'] >= 3].copy()
print(f"\n有效路口網格：{len(grid):,} 個")

grid['雨天敏感度'] = (grid['雨天比例'] / global_rain_rate).clip(0.5, 3.0)
grid['晚峰敏感度'] = (grid['晚峰比例'] / global_eve_rate).clip(0.5, 3.0)
grid['夜間敏感度'] = (grid['夜間比例'] / global_night_rate).clip(0.5, 3.0)
grid['早峰敏感度'] = (grid['早峰比例'] / global_peak_rate).clip(0.5, 3.0)
grid['機車敏感度'] = (grid['機車比例'] / max(global_moto_rate, 0.01)).clip(0.5, 3.0)

# 時間加權嚴重度
def time_weight(year):
    if year >= 2025:   return 1.0
    elif year >= 2023: return 0.7
    elif year >= 2021: return 0.4
    else:              return 0.2

df['time_w']      = df['發生年度_num'].apply(lambda x: time_weight(x) if pd.notna(x) else 0.2)
df['severity_w']  = df['嚴重度'].apply(lambda x: 10 if x==2 else 1)
df['weighted_score'] = df['time_w'] * df['severity_w']

weighted = df.groupby(['grid_lat','grid_lng'])['weighted_score'].sum().reset_index()
weighted.columns = ['grid_lat','grid_lng','加權分數']
grid = grid.merge(weighted, on=['grid_lat','grid_lng'], how='left')
grid['加權分數'] = grid['加權分數'].fillna(0)

# 早期事故密度（2020-2022）
df_early = df[df['發生年度_num'] <= 2022].copy()
early_grid = df_early.groupby(['grid_lat','grid_lng']).agg(
    早期A1密度=('嚴重度', lambda x: (x==2).sum()),
    早期A2密度=('嚴重度', lambda x: (x==1).sum()),
    早期事故密度=('嚴重度', 'count'),
).reset_index()
grid = grid.merge(early_grid, on=['grid_lat','grid_lng'], how='left')
grid[['早期A1密度','早期A2密度','早期事故密度']] = grid[['早期A1密度','早期A2密度','早期事故密度']].fillna(0)

# 風險等級
# 用百分位數定義風險等級（參考 Hazaymeh et al. 2022）
p75 = grid['加權分數'].quantile(0.75)  # 前25% = 高風險
p40 = grid['加權分數'].quantile(0.40)  # 40-75% = 中風險

def risk_label(row):
    s = row['加權分數']
    if s >= p75: return 2   # 高風險：前25%
    elif s >= p40: return 1  # 中風險：40-75%
    return 0                 # 低風險：後40%

grid['風險等級'] = grid.apply(risk_label, axis=1)
print(f"百分位數門檻：高風險≥{p75:.1f}，中風險≥{p40:.1f}")

grid['風險等級'] = grid.apply(risk_label, axis=1)
y = grid['風險等級']
print(f"\n高風險（加權分數≥15）：{(y==2).sum():,} 個")
print(f"中風險（加權分數5-14）：{(y==1).sum():,} 個")
print(f"低風險（加權分數<5） ：{(y==0).sum():,} 個")
print(f"\n加權分數最高：{grid['加權分數'].max():.1f}，平均：{grid['加權分數'].mean():.2f}")

Step 4：以路口網格聚合（50×50公尺）...
全市平均雨天比例：0.149
全市平均晚峰比例：0.215
全市平均夜間比例：0.134
全市平均早峰比例：0.208
全市平均機車比例：0.564

有效路口網格：5,386 個
百分位數門檻：高風險≥16.7，中風險≥5.1

高風險（加權分數≥15）：1,355 個
中風險（加權分數5-14）：1,878 個
低風險（加權分數<5） ：2,153 個

加權分數最高：503.0，平均：15.62


In [ ]:
print("Step 5-6：訓練 Random Forest（參考胡李2022：先切割再SMOTE）...")

feature_cols = ['雨天比例','有號誌','平均速限','晚峰比例','夜間比例',
                '早峰比例','機車比例','早期A1密度','早期A2密度','早期事故密度']

X = grid[feature_cols].fillna(0)
y = grid['風險等級']

# ① 先切割（75%訓練、25%測試，參考胡李2022）
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"訓練集（切割前）：{len(X_train_raw):,} 筆")
print(f"測試集：{len(X_test):,} 筆")
print(f"測試集風險分布（原始）：\n{y_test.value_counts().sort_index()}")

# ② 只對訓練集做 SMOTE（測試集保持原始不平衡分布）
sm = SMOTE(random_state=42, k_neighbors=3)
X_train, y_train = sm.fit_resample(X_train_raw, y_train_raw)
print(f"\n訓練集（SMOTE後）：{len(X_train):,} 筆")
print(f"訓練集風險分布（SMOTE後）：\n{pd.Series(y_train).value_counts().sort_index()}")

# ③ 訓練
clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
f1 = f1_score(y_test, y_pred, average='weighted')
a1_recall = recall_score(y_test, y_pred, labels=[2], average='macro')

print(f"\n整體 F1-score：{f1:.3f}")
print(f"高風險 Recall：{a1_recall:.3f}")
print(classification_report(y_test, y_pred, target_names=['低風險','中風險','高風險']))

feat_imp = dict(zip(feature_cols, clf.feature_importances_))
print("特徵重要度：")
for k,v in sorted(feat_imp.items(), key=lambda x:-x[1]):
    print(f"  {k}: {v:.3f}")

Step 5-6：訓練 Random Forest（參考胡李2022：先切割再SMOTE）...
訓練集（切割前）：4,039 筆
測試集：1,347 筆
測試集風險分布（原始）：
風險等級
0    538
1    470
2    339
Name: count, dtype: int64

訓練集（SMOTE後）：4,845 筆
訓練集風險分布（SMOTE後）：
風險等級
0    1615
1    1615
2    1615
Name: count, dtype: int64

整體 F1-score：0.812
高風險 Recall：0.817
              precision    recall  f1-score   support

         低風險       0.87      0.87      0.87       538
         中風險       0.73      0.74      0.73       470
         高風險       0.84      0.82      0.83       339

    accuracy                           0.81      1347
   macro avg       0.81      0.81      0.81      1347
weighted avg       0.81      0.81      0.81      1347

特徵重要度：
  早期事故密度: 0.183
  早期A2密度: 0.163
  雨天比例: 0.125
  夜間比例: 0.120
  晚峰比例: 0.113
  早峰比例: 0.100
  機車比例: 0.080
  平均速限: 0.058
  有號誌: 0.055
  早期A1密度: 0.003


In [ ]:
print("Step 7：計算風險分數（百分位數映射）...")
from scipy.stats import rankdata

def percentile_score(grid_df):
    result = pd.Series(index=grid_df.index, dtype=float)
    for level, lo, hi in [(2, 70, 100), (1, 40, 69), (0, 10, 39)]:
        mask = grid_df['風險等級'] == level
        if mask.sum() == 0:
            continue
        vals  = grid_df.loc[mask, '加權分數'].values.astype(float)
        ranks = rankdata(vals, method='average')
        normalized = (ranks - 1) / max(len(ranks) - 1, 1)
        scores = lo + normalized * (hi - lo)
        result.loc[mask] = np.round(scores, 1)
    return result

grid['風險分數'] = percentile_score(grid)

high_mask = grid['風險等級'] == 2
mid_mask  = grid['風險等級'] == 1
low_mask  = grid['風險等級'] == 0

print(f"高風險分數範圍：{grid[high_mask]['風險分數'].min():.1f} ~ {grid[high_mask]['風險分數'].max():.1f}")
print(f"中風險分數範圍：{grid[mid_mask]['風險分數'].min():.1f} ~ {grid[mid_mask]['風險分數'].max():.1f}")
print(f"低風險分數範圍：{grid[low_mask]['風險分數'].min():.1f} ~ {grid[low_mask]['風險分數'].max():.1f}")

# 確認重複程度
from collections import Counter
sc = Counter(round(s) for s in grid['風險分數'])
print(f"\n重複最多的分數（前5）：{sc.most_common(5)}")
print(f"不重複分數數量：{grid['風險分數'].nunique()} / {len(grid)}")

Step 7：計算風險分數（百分位數映射）...
高風險分數範圍：70.1 ~ 100.0
中風險分數範圍：40.1 ~ 68.9
低風險分數範圍：10.2 ~ 38.6

重複最多的分數（前5）：[(23, 203), (12, 148), (28, 147), (26, 128), (19, 122)]
不重複分數數量：467 / 5386


In [ ]:
print("Step 9：近期肇因分析（2023-2026年）...")
df['年度'] = pd.to_numeric(df['發生年度'], errors='coerce')
df_recent = df[df['年度'] >= 2023].copy()
print(f"近期資料：{len(df_recent):,} 筆")

CAUSE_HINTS = {
    '光線不足':   {'夜間':'此路口夜間照明不足，請開啟大燈並減速', '深夜':'深夜路段照明差，請特別注意來車'},
    '未開車燈':   {'夜間':'此路口夜間照明不足，請開啟大燈並減速'},
    '路面濕滑':   {'雨天':'路面易濕滑，請保持安全車距並減速'},
    '積水':       {'雨天':'此路段雨天易積水，請小心慢行'},
    '未禮讓行人': {'早峰':'行人穿越頻繁，請注意禮讓', '晚峰':'行人穿越頻繁，請注意禮讓'},
    '違規超速':   {'all':'此路口超速肇事頻繁，請注意速限'},
    '未注意車前': {'晚峰':'壅塞路段，請保持安全車距', '早峰':'通勤壅塞，請保持安全車距'},
    '闖紅燈':     {'all':'此路口闖紅燈事故多，請確認號誌再通行'},
    '酒駕':       {'夜間':'深夜路段注意對向來車異常行駛', '深夜':'深夜路段注意對向來車異常行駛'},
    '轉彎':       {'all':'此路口轉向事故頻繁，請確認來車再行駛'},
    '視線不良':   {'雨天':'視線受限，請減速慢行', '夜間':'視線受限，請開啟大燈'},
    '機車':       {'早峰':'機車流量高，請注意後照鏡', '晚峰':'機車流量高，請注意後照鏡'},
}

MOTO_CAUSE_HINTS = {
    '路面濕滑':   '雨天路面易打滑，機車請降速並避免急煞',
    '轉彎':       '此路口轉彎事故多，機車請提前減速確認來車',
    '未注意車前': '機車請保持車距，避免追撞',
    '違規超速':   '此路口機車超速事故頻繁，請注意速限',
    '闖紅燈':     '此路口闖燈事故多，機車請確認號誌',
    '未禮讓行人': '行人穿越頻繁，機車請減速禮讓',
    '光線不足':   '夜間照明不足，機車請開啟燈光確認路況',
    '變換車道':   '此路口車道變換事故多，機車請注意左右來車',
}

def get_hint(causes_text, period, weather):
    for keyword, mapping in CAUSE_HINTS.items():
        if keyword in str(causes_text):
            if weather == '雨天' and '雨天' in mapping:
                return mapping['雨天']
            elif period in mapping:
                return mapping[period]
            elif 'all' in mapping:
                return mapping['all']
    return None

def get_moto_hint(causes_text):
    for keyword, hint in MOTO_CAUSE_HINTS.items():
        if keyword in str(causes_text):
            return hint
    return None

def find_location(lat, lng, df_src, radius=0.0025):
    nearby = df_src[
        (abs(df_src['緯度']-lat) < radius) &
        (abs(df_src['經度']-lng) < radius)
    ]['發生地點']
    if len(nearby) == 0:
        return "桃園市路口", "桃園市"
    loc     = nearby.mode()[0]
    loc_str = str(loc)
    districts = ['中壢區','桃園區','平鎮區','八德區']
    dist = next((d for d in districts if d in loc_str), '桃園市')
    for remove in ['桃園市','桃園縣', dist]:
        loc_str = loc_str.replace(remove, '')
    return loc_str.strip()[:20], dist

def find_causes(lat, lng, df_src, radius=0.0025):
    nearby = df_src[
        (abs(df_src['緯度']-lat) < radius) &
        (abs(df_src['經度']-lng) < radius)
    ]
    if len(nearby) == 0:
        return [], {}, ''
    col = '肇因研判子類別名稱-主要'
    if col not in nearby.columns:
        return [], {}, ''
    top3   = nearby[col].dropna().value_counts().head(3)
    total  = max(len(nearby[col].dropna()), 1)
    causes = [[str(k)[:12], round(v/total*100)] for k,v in top3.items()]
    period_causes = {}
    for period in ['早峰','晚峰','夜間','深夜']:
        sub = nearby[nearby['時段']==period][col].dropna()
        if len(sub) >= 3:
            period_causes[period] = str(sub.mode()[0])
    rain_sub = nearby[nearby['天候']=='雨天'][col].dropna()
    if len(rain_sub) >= 3:
        period_causes['雨天'] = str(rain_sub.mode()[0])
    moto_nearby = nearby[nearby['涉及機車']==True] if '涉及機車' in nearby.columns else pd.DataFrame()
    moto_top = ''
    if len(moto_nearby) >= 3:
        mc = moto_nearby[col].dropna().value_counts()
        if len(mc) > 0:
            moto_top = str(mc.index[0])
    return causes, period_causes, moto_top

print("函數定義完成！")

Step 9：近期肇因分析（2023-2026年）...
近期資料：75,276 筆
函數定義完成！


In [ ]:
print("Step 10：輸出 risk_data.json...")

high = grid[grid['風險等級']==2].nlargest(114, '加權分數')
mid  = grid[grid['風險等級']==1].nlargest(76, '加權分數')
low  = grid[grid['風險等級']==0].nlargest(40,  '加權分數')
top_nodes = pd.concat([high, mid, low]).copy()

print(f"高風險：{len(high)}個，中風險：{len(mid)}個，低風險：{len(low)}個")
print("反查地址與肇因中（約需5-10分鐘）...")

nodes = []
for i, row in top_nodes.iterrows():
    s    = float(row['風險分數'])
    name, dist = find_location(row['grid_lat'], row['grid_lng'], df)
    causes, period_causes, moto_top = find_causes(
        row['grid_lat'], row['grid_lng'], df_recent
    )
    hints = {}
    for period in ['早峰','晚峰','夜間','深夜','雨天']:
        cause_text = period_causes.get(period, '')
        hint = get_hint(cause_text, period, '雨天' if period=='雨天' else '晴天')
        if hint:
            hints[period] = hint
    moto_hint = get_moto_hint(moto_top) if moto_top else ''

    nodes.append({
        "id":    len(nodes) + 1,
        "name":  name,
        "dist":  dist,
        "lat":   float(row['grid_lat']),
        "lng":   float(row['grid_lng']),
        "score": {
            "all":   s,
            "peak":  min(100, round(s * float(row['早峰敏感度']) * 1.05, 1)),
            "day":   round(s * 0.82, 1),
            "eve":   min(100, round(s * float(row['晚峰敏感度']) * 1.05, 1)),
            "night": min(100, round(s * float(row['夜間敏感度']) * 1.05, 1)),
        },
        "rain_sensitivity": round(float(row['雨天敏感度']), 2),
        "moto_sensitivity": round(float(row['機車敏感度']), 2),
        "acc":            int(row['事故總數']),
        "fatal":          int(row['A1件數']),
        "weighted_score": round(float(row['加權分數']), 1),
        "causes":         causes if causes else [["資料不足", 100]],
        "hints":          hints,
        "moto_hint":      moto_hint,
        "note": f"{'有' if row['有號誌']>0.5 else '無'}號誌，速限約{int(row['平均速限'])}km/h"
    })

output = {
    "nodes": nodes,
    "features": [
        {"name": k, "pct": round(v*100, 1)}
        for k,v in sorted(feat_imp.items(), key=lambda x:-x[1])
    ],
    "model_metrics": {
        "f1_score":        round(float(f1), 3),
        "a1_recall":       round(float(a1_recall), 3),
        "total_accidents": int(len(df)),
        "high_risk_count": int((grid['風險等級']==2).sum()),
        "mid_risk_count":  int((grid['風險等級']==1).sum()),
        "n_clusters":      0,
        "grid_size":       "50x50m",
        "data_years":      "2020-2026",
        "train_years":     "2020-2025（SMOTE擴增後）",
        "test_years":      "2020-2025（隨機20%）",
        "districts":       "中壢區、平鎮區、桃園區、八德區",
        "label_threshold": "加權分數≥15為高風險，5-14為中風險"
    },
    "generated_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
}

with open(f'{DATA_DIR}/risk_data.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\nrisk_data.json 完成！共 {len(nodes)} 個路口")
print(f"最高風險分數：{max(n['score']['all'] for n in nodes):.1f}")
print(f"最低風險分數：{min(n['score']['all'] for n in nodes):.1f}")
print(f"有情境提示的路口：{sum(1 for n in nodes if n['hints'])} 個")
print(f"有機車提示的路口：{sum(1 for n in nodes if n.get('moto_hint'))} 個")

Step 10：輸出 risk_data.json...
高風險：114個，中風險：76個，低風險：40個
反查地址與肇因中（約需5-10分鐘）...

risk_data.json 完成！共 230 個路口
最高風險分數：100.0
最低風險分數：38.6
有情境提示的路口：109 個
有機車提示的路口：21 個


In [ ]:
from google.colab import files
files.download(f'{DATA_DIR}/risk_data.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>